In [ ]:
import pandas as pd
import pyreadstat
import numpy as np
print("Imported successfully")


In [ ]:
# Load datasets
file_1999 = r"D:\Project_programming\case_study_mc\Data\raw\HSE_1999.dta"
file_2019 = r"D:\Project_programming\case_study_mc\Data\raw\HSE_2019.dta"

df99, meta99 = pyreadstat.read_dta(file_1999)
df19, meta19 = pyreadstat.read_dta(file_2019)

print("1999",df99.shape)
print("2019",df19.shape)

In [ ]:
# compare variable names
var99 = set(df99.columns)
var19 = set(df19.columns)

common_vars = var99.intersection(var19)
only_1999 = var99 - var19
only_2019 = var19 - var99

print("Common variables:", sorted(common_vars))
print("Only in 1999:", sorted(only_1999))
print("Only in 2019:", sorted(only_2019))

In [ ]:
# create a variable dictionary
file_path_dictionary = r"D:\Project_programming\case_study_mc\Data\processesed\variable_names.xlsx"

def variable_dictionary(df,meta,year):

  return pd.DataFrame({
    "year" : year,
    "Variable" : df.columns,
    "Label" : [
      meta.column_names_to_labels.get(col, "No label")
      for col in df.columns
    ],
    "dtype" : df.dtypes.astype(str).values
  })

dictionary = pd.concat([
  variable_dictionary(df99,meta99,1999),
  variable_dictionary(df19,meta19,2019)
])
# save to excel sheet
dictionary.to_excel(file_path_dictionary, index = False)
display(dictionary)


In [ ]:
# Canonical variable name : original variable name in each year

variable_map = {

    # 1. Smoking status
    "smoking_status": {
        1999: "cigst1",
        2019: "cigst1_19"
    },

    "ever_smoked_tobacco": {
        1999: "smkevr",
        2019: "smkevr_19"
    },

    "ever_smoked_cigarettes": {
        1999: "cigevr",
        2019: "cigevr_19"
    },

    "cigarette_smoking_intensity_band": {
        1999: "cigst2",
        2019: "cigst2_19"
    },

    "current_cigar_smoker": {
        1999: "cigarnow",
        2019: "CigarNow_19"
    },

    "current_pipe_smoker": {
        1999: "pipenowa",
        2019: "PipeNow_19"
    },

    # 2. Cigarette consumption
    "daily_cigarettes": {
        1999: "cigdyal",
        2019: "cigdyal"
    },

    "weekday_cigarettes": {
        1999: "cigwday",
        2019: "cigwday_19"
    },

    "weekend_cigarettes": {
        1999: "cigwend",
        2019: "cigwend_19"
    },

    "cigarette_type": {
        1999: "cigtyp",
        2019: "cigtyp_19"
    },

    # 3. Smoking history
    "age_started_regular_smoking": {
        1999: "startsmk",
        2019: "StartSmk19"
    },

    # 4. Demographic variables
    "age_years": {
        1999: "age",
        2019: None
    },

    "age_band": {
        1999: "ag16g10",
        2019: "age16g5"
    },

    "sex": {
        1999: "sex",
        2019: "Sex"
    },

    "region": {
        1999: "gor",
        2019: "GOR1"
    },

    # 5. Survey information
    "final_interview_outcome": {
        1999: "finoutc",
        2019: "FinOutc"
    },

    "individual_interview_outcome": {
        1999: "indout",
        2019: "IndOut"
    },

    "respondent_serial": {
        1999: "pserial",
        2019: "SerialA"
    },

    "household_serial": {
        1999: "hserial",
        2019: None
    },

    # 6. Survey weights: retain separately pending verification
    "standard_error_weight": {
        1999: "errorwt",
        2019: None
    },

    "scaled_error_weight": {
        1999: "scalewt",
        2019: None
    },

    "core_interview_weight": {
        1999: None,
        2019: "wt_int"
    }
}

In [ ]:
# automate renaming process
def standardize_variables(df,year,variable_map):
  
  # build mapping
  rename_dict = {
    original:standard
    for standard,year_mapping in variable_map.items()
    if(original := year_mapping.get(year)) is not None
    and original in df.columns
  }
  # rename variables
  result = df.rename(columns = rename_dict).copy() 

  return result

# standardize 1999
hse99 = standardize_variables(
  df99,
  1999,
  variable_map
)
# standardize 2019
hse19 = standardize_variables(
  df19,
  2019,
  variable_map
)

# check results
print("1999 variables:")
print(hse99.columns.tolist())

print("\n2019 variables:")
print(hse19.columns.tolist())

In [ ]:
# validae the mapping
def validate_mapping(df,year,variable_map):
  results = []
  for standard_name,year_mapping in variable_map.items():
    original_name = year_mapping.get(year)
    if original_name is None:
      status = "No equivalent identified"
    elif original_name in df.columns:
      status = "Found"
    else:
      status = "Missing from dataset"

    results.append({
      "Standard_name" : standard_name,
      "Original_name" : original_name,
      "Status" : status
    })
  return pd.DataFrame(results)

check99 = validate_mapping(df99,1999,variable_map)
check19 = validate_mapping(df19,2019,variable_map)
display(check99)
display(check19)

In [ ]:
# combine the two standardize datasets

# Add survey year
hse99['year'] = 1999
hse19['year'] = 2019

# Identify common columns
common_columns = (
  set(hse99.columns).intersection(hse19.columns)
)
common_columns = sorted(common_columns)

# combine data
hse_combined = pd.concat(
  [
    hse99[common_columns],
    hse19[common_columns]
  ],
  join = 'inner',
  ignore_index= True
)
# Display results
print(hse_combined.shape)
print(hse_combined.groupby("year").size())
print(hse_combined.columns.tolist())

# save to csv file
file_path_combined = r"D:\Project_programming\case_study_mc\Data\processesed/hse_data.csv"
hse_combined.to_csv(file_path_combined, index = False)


In [ ]:
# create a harmonization table excel sheet

file_path_mapping = r"D:\Project_programming\case_study_mc\Data\processesed/hse_table.xlsx"

mapping_table = pd.DataFrame([
  {
    "Standard_name" : standard,
    "Variable_1999" : mapping.get(1999),
    "Variable_2019" : mapping.get(2019)
  }
  for standard, mapping in variable_map.items()
])

display(mapping_table)
mapping_table.to_excel(file_path_mapping, index = False)